## Git importation and dependencies installation

In [ ]:
!git clone https://github.com/brakoto/LOG6305_tp4.git

Cloning into 'LOG6305_tp4'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 104 (delta 48), reused 77 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 344.13 KiB | 9.30 MiB/s, done.
Resolving deltas: 100% (48/48), done.


In [ ]:
%cd LOG6305_tp4/

/content/LOG6305_tp4


In [ ]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.2/254.2 kB 9.6 MB/s eta 0:00:00


In [ ]:
!pip install -U bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00


## Import depedencies

In [ ]:
import importlib
from poly_llm.common.abstract_executor import AbstractExecutor
from poly_llm.common.prompt_generator import PromptGenerator
from poly_llm.generators.llm_test_generator import LLMTestGenerator
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

## Model Loading

In [ ]:
from poly_llm.common.prompt_generator import PromptGenerator
from poly_llm.generators.llm_test_generator import LLMTestGenerator
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(model_name,
      quantization_config=bnb_config,
      device_map="auto", trust_remote_code=True,
      dtype=torch.bfloat16
)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
def create_test_function(tested_file, few_shot_examples=[]):
  prompt_generator = PromptGenerator(tested_file)
  llm_generator = LLMTestGenerator(model, tokenizer, tested_file)
  if few_shot_examples:
    prompt = prompt_generator.generate_prompt(few_shot_examples=few_shot_examples)
  else:
    prompt = prompt_generator.generate_prompt()
  print(prompt)
  test, test_name = llm_generator.create_test_function(prompt)
  return test, test_name


In [ ]:
def save_test_to_file(test, tested_file,filename="test.py"):
  llm_generator = LLMTestGenerator(model, tokenizer, tested_file)
  llm_generator.write_test_to_file(test, filename=filename)

In [ ]:
def coverage_llm_produced_code(filename, tested_function, test_name):
  module_name = filename.split(".")[0]
  function_name = test_name

  module = importlib.import_module(module_name)
  function = getattr(module, function_name)
  executor2 = AbstractExecutor(function)

  coverage_data = executor2._execute_input(tested_function)
  return coverage_data

In [ ]:
def delete_model(model, tokenizer):
  del model
  del tokenizer
  torch.cuda.empty_cache()

## `find_closest_elements` Test generation

In [ ]:
from poly_llm.to_test.find_closest_elements import find_closest_elements

### Measure of the initial coverage (`find_closest_elements`)

In [ ]:
find_closest_elements_executor = AbstractExecutor(find_closest_elements)
inputs = [
    [1.0, 2.0, 3.9, 4.0, 5.0, 2.2],
    [1.0, 2.0, 5.9, 4.0, 5.0]
]
total_coverage_report = []
for input in inputs:
    coverage_data = find_closest_elements_executor._execute_input(input)
    total_coverage_report.append(coverage_data["coverage"])
    #print(f"{input}: {coverage_data}")
total_coverage_report

max_line_coverage = max([total_coverage_report[i]['percent_covered'] for i in range(len(inputs))])
max_branch_coverage = max([total_coverage_report[i]['percent_branches_covered'] for i in range(len(inputs))])
print(f"Initial line coverage: {max_line_coverage}")
print(f"Initial branch coverage: {max_branch_coverage}")

Initial line coverage: 92.0
Initial branch coverage: 100.0


### Zero-shot Test Generation with the LLM (`find_closest_elements`)

In [ ]:
# Creation of the test suite
zero_find_closest_elements_test, zero_find_closest_elements_test_name = create_test_function(find_closest_elements)

# Save the test suite in the dedicated file
zero_find_closest_elements_test_filename = "zero_test_find_closest_elements.py"
save_test_to_file(zero_find_closest_elements_test, find_closest_elements, filename=zero_find_closest_elements_test_filename)

Generate unit tests for the provided input file.
Input
def find_closest_elements(numbers: List[float]) -> Tuple[float, float]:
    
    closest_pair = None
    distance = None

    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                if distance is None:
                    distance = abs(elem - elem2)
                    closest_pair = tuple(sorted([elem, elem2]))
                else:
                    new_distance = abs(elem - elem2)
                    if new_distance < distance:
                        distance = new_distance
                        closest_pair = tuple(sorted([elem, elem2]))

    return closest_pair


Test function written to zero_test_find_closest_elements.py


### Few-shot Test Generation with the LLM (`find_closest_elements`)

In [ ]:
few_shot_examples = ['''def test_find_closest_elements(): \n
    assert find_closest_elements([1.0, 2.0, 3.9, 4.0, 5.0, 2.2]) == (3.9, 4.0) \n
    assert find_closest_elements([1.0, 2.0, 5.9, 4.0, 5.0]) == (5.0, 5.9) \n''']
few_find_closest_elements_test, few_find_closest_elements_test_name = create_test_function(find_closest_elements, few_shot_examples=few_shot_examples)

# Save the test suite in the dedicated file
few_find_closest_elements_test_filename = "few_test_find_closest_elements.py"
save_test_to_file(few_find_closest_elements_test, find_closest_elements, filename=few_find_closest_elements_test_filename)

Generate unit tests for the provided input file.
Input
def find_closest_elements(numbers: List[float]) -> Tuple[float, float]:
    
    closest_pair = None
    distance = None

    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                if distance is None:
                    distance = abs(elem - elem2)
                    closest_pair = tuple(sorted([elem, elem2]))
                else:
                    new_distance = abs(elem - elem2)
                    if new_distance < distance:
                        distance = new_distance
                        closest_pair = tuple(sorted([elem, elem2]))

    return closest_pair

Below is a list of test case examples.
Example
def test_find_closest_elements(): 

    assert find_closest_elements([1.0, 2.0, 3.9, 4.0, 5.0, 2.2]) == (3.9, 4.0) 

    assert find_closest_elements([1.0, 2.0, 5.9, 4.0, 5.0]) == (5.0, 5.9) 


Test function written to few_test_find_closest_elem

### Generated test coverage (`find_closest_elements`)

In [ ]:
zero_find_closest_elements_llm_test_coverage = coverage_llm_produced_code(zero_find_closest_elements_test_filename, find_closest_elements, zero_find_closest_elements_test_name)
print(zero_find_closest_elements_llm_test_coverage)
few_find_closest_elements_llm_test_coverage = coverage_llm_produced_code(few_find_closest_elements_test_filename, find_closest_elements, few_find_closest_elements_test_name)
print(few_find_closest_elements_llm_test_coverage)

zero_line_cov = zero_find_closest_elements_llm_test_coverage['coverage'].get('percent_covered', 'N/A (Test failed or no coverage data)')
zero_branch_cov = zero_find_closest_elements_llm_test_coverage['coverage'].get('percent_branches_covered', 'N/A (Test failed or no coverage data)')
few_line_cov = few_find_closest_elements_llm_test_coverage['coverage'].get('percent_covered', 'N/A (Test failed or no coverage data)')
few_branch_cov = few_find_closest_elements_llm_test_coverage['coverage'].get('percent_branches_covered', 'N/A (Test failed or no coverage data)')

print(f"Zero-shot Line coverage: {zero_line_cov}")
print(f"Zero-shot Branch coverage: {zero_branch_cov}")
print(f"Few-shot Line coverage: {few_line_cov}")
print(f"Few-shot Branch coverage: {few_branch_cov}")

zero_test_find_closest_elements
test_find_closest_elements
<function test_find_closest_elements at 0x7b6c21c0cd60>
<function find_closest_elements at 0x7b6c21c0c0e0>
{'input': <function find_closest_elements at 0x7b6c21c0c0e0>, 'exceptions': 1, 'execution_time': 0.0001437664031982422, 'coverage': {}}
few_test_find_closest_elements
test_find_closest_elements
<function test_find_closest_elements at 0x7b6c21c0cb80>
<function find_closest_elements at 0x7b6c21c0c0e0>
{'input': <function find_closest_elements at 0x7b6c21c0c0e0>, 'exceptions': 1, 'execution_time': 0.0002601146697998047, 'coverage': {}}
Zero-shot Line coverage: N/A (Test failed or no coverage data)
Zero-shot Branch coverage: N/A (Test failed or no coverage data)
Few-shot Line coverage: N/A (Test failed or no coverage data)
Few-shot Branch coverage: N/A (Test failed or no coverage data)


## `numerical_letter_grade` Test generation

In [ ]:
from poly_llm.to_test.numerical_letter_grade import numerical_letter_grade

### Measure of the initial coverage (`numerical_letter_grade`)

In [ ]:
numerical_letter_grade_executor = AbstractExecutor(numerical_letter_grade)

inputs = [
    [4.0, 3, 1.7, 2, 3.5],
    [1.2]
]
total_coverage_report = []
for input in inputs:
    coverage_data = numerical_letter_grade_executor._execute_input(input)
    total_coverage_report.append(coverage_data["coverage"])
    #print(f"{input}: {coverage_data}")
total_coverage_report

max_line_coverage = max([total_coverage_report[i]['percent_covered'] for i in range(len(inputs))])
max_branch_coverage = max([total_coverage_report[i]['percent_branches_covered'] for i in range(len(inputs))])
print(f"Initial line coverage: {max_line_coverage}")
print(f"Initial branch coverage: {max_branch_coverage}")

Initial line coverage: 58.18181818181818
Initial branch coverage: 57.69230769230769


### Zero-shot Test Generation with the LLM (`numerical_letter_grade`)

In [ ]:
# Creation of the test suite
zero_numerical_letter_grade_test, zero_numerical_letter_grade_test_name = create_test_function(numerical_letter_grade)

# Save the test suite in the dedicated file
zero_numerical_letter_grade_test_filename = "zero_test_numerical_letter_grade.py"
save_test_to_file(zero_numerical_letter_grade_test, numerical_letter_grade, filename=zero_numerical_letter_grade_test_filename)

Generate unit tests for the provided input file.
Input
def numerical_letter_grade(grades):
    

   
    letter_grade = []
    for gpa in grades:
        if gpa == 4.0:
            letter_grade.append("A+")
        elif gpa > 3.7:
            letter_grade.append("A")
        elif gpa > 3.3:
            letter_grade.append("A-")
        elif gpa > 3.0:
            letter_grade.append("B+")
        elif gpa > 2.7:
            letter_grade.append("B")
        elif gpa > 2.3:
            letter_grade.append("B-")
        elif gpa > 2.0:
            letter_grade.append("C+")
        elif gpa > 1.7:
            letter_grade.append("C")
        elif gpa > 1.3:
            letter_grade.append("C-")
        elif gpa > 1.0:
            letter_grade.append("D+")
        elif gpa > 0.7:
            letter_grade.append("D")
        elif gpa > 0.0:
            letter_grade.append("D-")
        else:
            letter_grade.append("E")
    return letter_grade


Test function written to zero_test_num

### Few-shot Test Generation with the LLM (`numerical_letter_grade`)

In [ ]:

few_shot_examples = ['''def test_numerical_letter_grade(): \n
    assert numerical_letter_grade([4.0, 3, 1.7, 2, 3.5]) == ['A+', 'B', 'C-', 'C', 'A-'] \n
    assert numerical_letter_grade([1.2]) == ['D+'] \n''']
few_numerical_letter_grade_test, few_numerical_letter_grade_test_name = create_test_function(numerical_letter_grade, few_shot_examples=few_shot_examples)

# Save the test suite in the dedicated file
few_numerical_letter_grade_test_filename = "few_test_numerical_letter_grade.py"
save_test_to_file(few_numerical_letter_grade_test, numerical_letter_grade, filename=few_numerical_letter_grade_test_filename)

Generate unit tests for the provided input file.
Input
def numerical_letter_grade(grades):
    

   
    letter_grade = []
    for gpa in grades:
        if gpa == 4.0:
            letter_grade.append("A+")
        elif gpa > 3.7:
            letter_grade.append("A")
        elif gpa > 3.3:
            letter_grade.append("A-")
        elif gpa > 3.0:
            letter_grade.append("B+")
        elif gpa > 2.7:
            letter_grade.append("B")
        elif gpa > 2.3:
            letter_grade.append("B-")
        elif gpa > 2.0:
            letter_grade.append("C+")
        elif gpa > 1.7:
            letter_grade.append("C")
        elif gpa > 1.3:
            letter_grade.append("C-")
        elif gpa > 1.0:
            letter_grade.append("D+")
        elif gpa > 0.7:
            letter_grade.append("D")
        elif gpa > 0.0:
            letter_grade.append("D-")
        else:
            letter_grade.append("E")
    return letter_grade

Below is a list of test case examples.


### Generated test coverage (`numerical_letter_grade`)

In [ ]:
zero_numerical_letter_grade_llm_test_coverage = coverage_llm_produced_code(zero_numerical_letter_grade_test_filename, numerical_letter_grade, zero_numerical_letter_grade_test_name)
print(zero_numerical_letter_grade_llm_test_coverage)
few_numerical_letter_grade_llm_test_coverage = coverage_llm_produced_code(few_numerical_letter_grade_test_filename, numerical_letter_grade, few_numerical_letter_grade_test_name)
print(few_numerical_letter_grade_llm_test_coverage)

zero_line_cov = zero_numerical_letter_grade_llm_test_coverage['coverage'].get('percent_covered', 'N/A (Test failed or no coverage data)')
zero_branch_cov = zero_numerical_letter_grade_llm_test_coverage['coverage'].get('percent_branches_covered', 'N/A (Test failed or no coverage data)')
few_line_cov = few_numerical_letter_grade_llm_test_coverage['coverage'].get('percent_covered', 'N/A (Test failed or no coverage data)')
few_branch_cov = few_numerical_letter_grade_llm_test_coverage['coverage'].get('percent_branches_covered', 'N/A (Test failed or no coverage data)')

print(f"Zero-shot Line coverage: {zero_line_cov}")
print(f"Zero-shot Branch coverage: {zero_branch_cov}")
print(f"Few-shot Line coverage: {few_line_cov}")
print(f"Few-shot Branch coverage: {few_branch_cov}")

zero_test_numerical_letter_grade
test_numerical_letter_grade
<function test_numerical_letter_grade at 0x7b6c21c0d300>
<function numerical_letter_grade at 0x7b6c21c0cf40>
{'input': <function numerical_letter_grade at 0x7b6c21c0cf40>, 'exceptions': 1, 'execution_time': 8.916854858398438e-05, 'coverage': {}}
few_test_numerical_letter_grade
test_numerical_letter_grade
<function test_numerical_letter_grade at 0x7b6c21c0cfe0>
<function numerical_letter_grade at 0x7b6c21c0cf40>
{'input': <function numerical_letter_grade at 0x7b6c21c0cf40>, 'exceptions': 1, 'execution_time': 0.00023603439331054688, 'coverage': {}}
Zero-shot Line coverage: N/A (Test failed or no coverage data)
Zero-shot Branch coverage: N/A (Test failed or no coverage data)
Few-shot Line coverage: N/A (Test failed or no coverage data)
Few-shot Branch coverage: N/A (Test failed or no coverage data)


## `separate_paren_groups` Test generation

In [ ]:
from poly_llm.to_test.separate_paren_groups import separate_paren_groups

### Measure of the initial coverage (`separate_paren_groups`)

In [ ]:
separate_paren_groups_executor = AbstractExecutor(separate_paren_groups)

inputs = [
    '(()()) ((())) () ((())()())',
    '() (()) ((())) (((())))'
]
total_coverage_report = []
for input in inputs:
    coverage_data = separate_paren_groups_executor._execute_input(input)
    total_coverage_report.append(coverage_data["coverage"])
    #print(f"{input}: {coverage_data}")
total_coverage_report

max_line_coverage = max([total_coverage_report[i]['percent_covered'] for i in range(len(inputs))])
max_branch_coverage = max([total_coverage_report[i]['percent_branches_covered'] for i in range(len(inputs))])
print(f"Initial line coverage: {max_line_coverage}")
print(f"Initial branch coverage: {max_branch_coverage}")

Initial line coverage: 91.66666666666667
Initial branch coverage: 100.0


### Zero-shot Test Generation with the LLM (`separate_paren_groups`)

In [ ]:
# Creation of the test suite
zero_separate_paren_groups_test, zero_separate_paren_groups_test_name = create_test_function(separate_paren_groups)

# Save the test suite in the dedicated file
zero_separate_paren_groups_test_filename = "zero_test_separate_paren_groups.py"
save_test_to_file(zero_separate_paren_groups_test, separate_paren_groups, filename=zero_separate_paren_groups_test_filename)

Generate unit tests for the provided input file.
Input
def separate_paren_groups(paren_string: str) -> List[str]:
    
    result = []
    current_string = []
    current_depth = 0

    for c in paren_string:
        if c == '(':
            current_depth += 1
            current_string.append(c)
        elif c == ')':
            current_depth -= 1
            current_string.append(c)

            if current_depth == 0:
                result.append(''.join(current_string))
                current_string.clear()

    return result


Test function written to zero_test_separate_paren_groups.py


### Few-shot Test Generation with the LLM (`separate_paren_groups`)

In [ ]:

few_shot_examples = ['''def test_separate_paren_groups(): \n
    assert separate_paren_groups('(()()) ((())) () ((())()())') == [
        '(()())', '((()))', '()', '((())()())'
    ] \n
    assert separate_paren_groups('() (()) ((())) (((())))') == [
        '()', '(())', '((()))', '(((())))'
    ] \n''']
few_separate_paren_groups_test, few_separate_paren_groups_test_name = create_test_function(separate_paren_groups, few_shot_examples=few_shot_examples)

# Save the test suite in the dedicated file
few_separate_paren_groups_test_filename = "few_test_separate_paren_group.py"
save_test_to_file(few_separate_paren_groups_test, separate_paren_groups, filename=few_separate_paren_groups_test_filename)

Generate unit tests for the provided input file.
Input
def separate_paren_groups(paren_string: str) -> List[str]:
    
    result = []
    current_string = []
    current_depth = 0

    for c in paren_string:
        if c == '(':
            current_depth += 1
            current_string.append(c)
        elif c == ')':
            current_depth -= 1
            current_string.append(c)

            if current_depth == 0:
                result.append(''.join(current_string))
                current_string.clear()

    return result

Below is a list of test case examples.
Example
def test_separate_paren_groups(): 

    assert separate_paren_groups('(()()) ((())) () ((())()())') == [
        '(()())', '((()))', '()', '((())()())'
    ] 

    assert separate_paren_groups('() (()) ((())) (((())))') == [
        '()', '(())', '((()))', '(((())))'
    ] 


Test function written to few_test_separate_paren_group.py


### Generated test coverage (`separate_paren_groups`)

In [ ]:
zero_separate_paren_group_llm_test_coverage = coverage_llm_produced_code(zero_separate_paren_groups_test_filename, separate_paren_groups, zero_separate_paren_groups_test_name)
print(zero_separate_paren_group_llm_test_coverage)
few_separate_paren_group_llm_test_coverage = coverage_llm_produced_code(few_separate_paren_groups_test_filename, separate_paren_groups, few_separate_paren_groups_test_name)
print(few_separate_paren_group_llm_test_coverage)

zero_line_cov = zero_separate_paren_group_llm_test_coverage['coverage'].get('percent_covered', 'N/A (Test failed or no coverage data)')
zero_branch_cov = zero_separate_paren_group_llm_test_coverage['coverage'].get('percent_branches_covered', 'N/A (Test failed or no coverage data)')
few_line_cov = few_separate_paren_group_llm_test_coverage['coverage'].get('percent_covered', 'N/A (Test failed or no coverage data)')
few_branch_cov = few_separate_paren_group_llm_test_coverage['coverage'].get('percent_branches_covered', 'N/A (Test failed or no coverage data)')

print(f"Zero-shot Line coverage: {zero_line_cov}")
print(f"Zero-shot Branch coverage: {zero_branch_cov}")
print(f"Few-shot Line coverage: {few_line_cov}")
print(f"Few-shot Branch coverage: {few_branch_cov}")

zero_test_separate_paren_groups
test_separate_paren_groups
<function test_separate_paren_groups at 0x7b6c21c0dee0>
<function separate_paren_groups at 0x7b6c21c0d760>
{'input': <function separate_paren_groups at 0x7b6c21c0d760>, 'exceptions': 1, 'execution_time': 0.00021791458129882812, 'coverage': {}}
few_test_separate_paren_group
test_separate_paren_groups
<function test_separate_paren_groups at 0x7b6c21c0f9c0>
<function separate_paren_groups at 0x7b6c21c0d760>
{'input': <function separate_paren_groups at 0x7b6c21c0d760>, 'exceptions': 0, 'execution_time': 0.0002608299255371094, 'coverage': {'covered_lines': 24, 'num_statements': 27, 'percent_covered': 91.42857142857143, 'percent_covered_display': '91', 'missing_lines': 3, 'excluded_lines': 7, 'percent_statements_covered': 88.88888888888889, 'percent_statements_covered_display': '89', 'num_branches': 8, 'num_partial_branches': 0, 'covered_branches': 8, 'missing_branches': 0, 'percent_branches_covered': 100.0, 'percent_branches_covered

## `file_name_check` Test generation

In [ ]:
from poly_llm.to_test.file_name_check import file_name_check

### Measure of the initial coverage

In [ ]:
file_name_executor = AbstractExecutor(file_name_check)
inputs = [
    "example.txt",
    "1example.dll",
    ".txt",
]
total_coverage_report = []
for input in inputs:
    coverage_data = file_name_executor._execute_input(input)
    total_coverage_report.append(coverage_data["coverage"])
    #print(f"{input}: {coverage_data}")
total_coverage_report

max_line_coverage = max([total_coverage_report[i]['percent_covered'] for i in range(len(inputs))])
max_branch_coverage = max([total_coverage_report[i]['percent_branches_covered'] for i in range(len(inputs))])
print(f"Initial line coverage: {max_line_coverage}")
print(f"Initial branch coverage: {max_branch_coverage}")

Initial line coverage: 56.0
Initial branch coverage: 50.0


### Zero-shot Test Generation with the LLM

In [ ]:
# Creation of the test suite
zero_file_name_check_test, zero_file_name_check_test_name = create_test_function(file_name_check)

# Save the test suite in the dedicated file
zero_file_name_check_test_filename = "zero_test_file_name_check.py"
save_test_to_file(zero_file_name_check_test, file_name_check, filename=zero_file_name_check_test_filename)

Generate unit tests for the provided input file.
Input
def file_name_check(file_name):
    
    suf = ['txt', 'exe', 'dll']
    lst = file_name.split(sep='.')
    if len(lst) != 2:
        return 'No'
    if not lst[1] in suf:
        return 'No'
    if len(lst[0]) == 0:
        return 'No'
    if not lst[0][0].isalpha():
        return 'No'
    t = len([x for x in lst[0] if x.isdigit()])
    if t > 3:
        return 'No'
    return 'Yes'


Test function written to zero_test_file_name_check.py


### Few-shot Test Generation with the LLM

In [ ]:
# Creation of the test suite
few_shot_examples = ['''def test_file_name_check(): \n
    assert file_name_check("example.txt") == 'Yes'  \n
    assert file_name_check("1example.dll") == 'No' \n
    assert file_name_check('.txt') == 'No' \n''']
few_file_name_check_test, few_file_name_check_test_name = create_test_function(file_name_check, few_shot_examples=few_shot_examples)

# Save the test suite in the dedicated file
few_file_name_check_test_filename = "few_test_file_name_check.py"
save_test_to_file(few_file_name_check_test, file_name_check, filename=few_file_name_check_test_filename)

Generate unit tests for the provided input file.
Input
def file_name_check(file_name):
    
    suf = ['txt', 'exe', 'dll']
    lst = file_name.split(sep='.')
    if len(lst) != 2:
        return 'No'
    if not lst[1] in suf:
        return 'No'
    if len(lst[0]) == 0:
        return 'No'
    if not lst[0][0].isalpha():
        return 'No'
    t = len([x for x in lst[0] if x.isdigit()])
    if t > 3:
        return 'No'
    return 'Yes'

Below is a list of test case examples.
Example
def test_file_name_check(): 

    assert file_name_check("example.txt") == 'Yes'  

    assert file_name_check("1example.dll") == 'No' 

    assert file_name_check('.txt') == 'No' 


Test function written to few_test_file_name_check.py


### Generated test coverage

In [ ]:
zero_file_name_check_llm_test_coverage = coverage_llm_produced_code(zero_file_name_check_test_filename, file_name_check, zero_file_name_check_test_name)
print(zero_file_name_check_llm_test_coverage)
few_file_name_check_llm_test_coverage = coverage_llm_produced_code(few_file_name_check_test_filename, file_name_check, few_file_name_check_test_name)
print(few_file_name_check_llm_test_coverage)

print(f"Zero-shot Line coverage: {zero_file_name_check_llm_test_coverage['coverage']['percent_covered']}")
print(f"Zero-shot Branch coverage: {zero_file_name_check_llm_test_coverage['coverage']['percent_branches_covered']}")
print(f"Few-shot Line coverage: {few_file_name_check_llm_test_coverage['coverage']['percent_covered']}")
print(f"Few-shot Branch coverage: {few_file_name_check_llm_test_coverage['coverage']['percent_branches_covered']}")

## `closest_integer` Test generation

In [ ]:
from poly_llm.to_test.closest_integer import closest_integer

### Measure of the initial coverage

In [ ]:
closest_integer_executor = AbstractExecutor(closest_integer)
inputs = [
    "10",
    "14.5"
]
total_coverage_report = []
for input in inputs:
    coverage_data = closest_integer_executor._execute_input(input)
    total_coverage_report.append(coverage_data["coverage"])
    #print(f"{input}: {coverage_data}")
total_coverage_report

max_line_coverage = max([total_coverage_report[i]['percent_covered'] for i in range(len(inputs))])
max_branch_coverage = max([total_coverage_report[i]['percent_branches_covered'] for i in range(len(inputs))])
print(f"Initial line coverage: {max_line_coverage}")
print(f"Initial branch coverage: {max_branch_coverage}")

Initial line coverage: 50.0
Initial branch coverage: 40.0


### Zero-shot Test Generation with the LLM

In [ ]:
# Creation of the test suite
zero_closest_integer_test, zero_closest_integer_test_name = create_test_function(closest_integer)

# Save the test suite in the dedicated file
zero_closest_integer_test_filename = "zero_test_closest_integer.py"
save_test_to_file(zero_closest_integer_test, closest_integer, filename=zero_closest_integer_test_filename)

Generate unit tests for the provided input file.
Input
def closest_integer(value):
    '''
    Create a function that takes a value (string) representing a number
    and returns the closest integer to it. If the number is equidistant
    from two integers, round it away from zero.
    Note:
    Rounding away from zero means that if the given number is equidistant
    from two integers, the one you should return is the one that is the
    farthest from zero. For example closest_integer("14.5") should
    return 15 and closest_integer("-14.5") should return -15.
    '''
    from math import floor, ceil

    if value.count('.') == 1:
        # remove trailing zeros
        while (value[-1] == '0'):
            value = value[:-1]

    num = float(value)
    if value[-2:] == '.5':
        if num > 0:
            res = ceil(num)
        else:
            res = floor(num)
    elif len(value) > 0:
        res = int(round(num))
    else:
        res = 0

    return res


Test function written 

### Few-shot Test Generation with the LLM

In [ ]:
# Creation of the test suite
few_shot_examples = ['''def test_closest_integer(): \n
    assert closest_integer("10") == 10, \n
    assert closest_integer("14.5") == 15 \n''']
few_closest_integer_test, few_closest_integer_test_name = create_test_function(closest_integer, few_shot_examples=few_shot_examples)

# Save the test suite in the dedicated file
few_closest_integer_test_filename = "few_test_closest_integer.py"
save_test_to_file(few_closest_integer_test, closest_integer, filename=few_closest_integer_test_filename)

Generate unit tests for the provided input file.
Input
def closest_integer(value):
    '''
    Create a function that takes a value (string) representing a number
    and returns the closest integer to it. If the number is equidistant
    from two integers, round it away from zero.
    Note:
    Rounding away from zero means that if the given number is equidistant
    from two integers, the one you should return is the one that is the
    farthest from zero. For example closest_integer("14.5") should
    return 15 and closest_integer("-14.5") should return -15.
    '''
    from math import floor, ceil

    if value.count('.') == 1:
        # remove trailing zeros
        while (value[-1] == '0'):
            value = value[:-1]

    num = float(value)
    if value[-2:] == '.5':
        if num > 0:
            res = ceil(num)
        else:
            res = floor(num)
    elif len(value) > 0:
        res = int(round(num))
    else:
        res = 0

    return res

Below is a list of test

### Generated test coverage

In [ ]:
zero_closest_integer_llm_test_coverage = coverage_llm_produced_code(zero_closest_integer_test_filename, closest_integer, zero_closest_integer_test_name)
print(zero_closest_integer_llm_test_coverage)
few_closest_integer_llm_test_coverage = coverage_llm_produced_code(few_closest_integer_test_filename, closest_integer, few_closest_integer_test_name)
print(few_file_name_check_llm_test_coverage)

print(f"Zero-shot Line coverage: {zero_closest_integer_llm_test_coverage['coverage']['percent_covered']}")
print(f"Zero-shot Branch coverage: {zero_file_name_check_llm_test_coverage['coverage']['percent_branches_covered']}")
print(f"Few-shot Line coverage: {few_closest_integer_llm_test_coverage['coverage']['percent_covered']}")
print(f"Few-shot Branch coverage: {few_closest_integer_llm_test_coverage['coverage']['percent_branches_covered']}")